# W9-D5 概念实验：DigitalEmployeeDefinition 为什么不拥有 Runtime？

配套阅读：同名 `.md`。这里不复述阅读材料，而是用小规模、可运行的模型检验其中的架构约束。

## 实验问题

**问题 1：同一数字员工定义能否拥有多个独立的部署状态？**

定义只表达“谁、做什么、引用什么”；每个 mall/environment 的运行状态由独立 Deployment 保存。

In [ ]:
from dataclasses import dataclass, field, replace
from hashlib import sha256
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(202608)
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()
@dataclass(frozen=True)
class DigitalEmployeeDefinition:
    employee_id: str
    definition_version: int
    contract_digest: str
    blueprint_digest: str
    publish_scope: tuple

@dataclass
class Deployment:
    deployment_id: str
    employee_id: str
    environment: str
    state: str

employee = DigitalEmployeeDefinition("contract-reviewer", 1, "sha256:contract", "sha256:blueprint", ("shanghai", "beijing"))
sh = Deployment("dep-sh", employee.employee_id, "shanghai-prod", "active")
bj = Deployment("dep-bj", employee.employee_id, "beijing-prod", "suspended")
print(employee)
print("部署状态：", [(d.environment, d.state) for d in [sh, bj]])
assert employee.employee_id == sh.employee_id == bj.employee_id and sh.state != bj.state

## 实验问题

**问题 2：如果把 Runtime 状态塞回定义，会发生什么耦合？**

比较“定义含一个 state”与“定义引用多个 Deployment”：前者无法同时表达上海在服役、北京暂停。

In [ ]:
@dataclass
class BadEmployee:
    employee_id: str
    runtime_state: str
bad = BadEmployee("contract-reviewer", "active")
requested = {"shanghai": "active", "beijing": "suspended"}
print("错误模型只能存一个 runtime_state：", bad.runtime_state)
print("实际需要的环境状态：", requested)
assert len(set(requested.values())) > 1
print("结论：多部署需求迫使运行生命周期离开定义对象。")

## 实验问题

**问题 3：知识或策略变更为何不应修改员工身份？**

把知识更新限定为上海 DeploymentRevision 的变化，员工定义与北京运行闭包保持不变。

In [ ]:
@dataclass(frozen=True)
class RuntimeClosure:
    deployment_id: str
    knowledge_digest: str
    policy_digest: str

sh_v1 = RuntimeClosure(sh.deployment_id, "sha256:kb-sh-v1", "sha256:policy-sh")
sh_v2 = replace(sh_v1, knowledge_digest="sha256:kb-sh-v2")
bj_v1 = RuntimeClosure(bj.deployment_id, "sha256:kb-bj-v1", "sha256:policy-bj")
print("定义版本仍为：", employee.definition_version)
print("上海知识：", sh_v1.knowledge_digest, "->", sh_v2.knowledge_digest)
print("北京知识：", bj_v1.knowledge_digest)
assert employee.definition_version == 1 and bj_v1.knowledge_digest != sh_v2.knowledge_digest

## 实验问题

**问题 4：分离后，变更影响范围如何缩小？**

以 10 个商场为例，对比“修改定义即全体受影响”和“只创建一个部署 revision”的影响数量。

In [ ]:
malls = [f"mall-{i}" for i in range(10)]
definition_coupled = len(malls)
deployment_scoped = 1
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(["定义拥有 Runtime", "独立 DeploymentRevision"], [definition_coupled, deployment_scoped], color=["#d95f02", "#1b9e77"])
ax.set_ylabel("一次知识更新影响的商场数"); ax.set_title("定义/运行分离缩小变更爆炸半径")
plt.tight_layout(); plt.show()
print("员工身份稳定，运行调整按环境发生。")